The implementation of **PPO (Proximal Policy Optimization)** algorithm.

Article Name: **Proximal Policy Optimization Algorithms**

link: https://arxiv.org/abs/1707.06347

In [18]:
# Dependencies Installation & Setup

!pip install -q swig
!pip install -q "gymnasium[box2d,mujoco,atari,accept-rom-license]"
!pip install -q stable-baselines3[extra]
!pip install -q matplotlib tensorboard

import os
import time
import random
import numpy as np
import torch as T
import torch.nn as nn
import torch.optim as optim
from torch.distributions.categorical import Categorical
from torch.distributions.normal import Normal
from torch.utils.tensorboard import SummaryWriter
import gymnasium as gym
import matplotlib.pyplot as plt

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    T.manual_seed(seed)
    T.cuda.manual_seed(seed)
    T.cuda.manual_seed_all(seed)
    T.backends.cudnn.deterministic = True

seed_everything(42)
print("Environment successfully initialized!")

Exception ignored in: <_io.BytesIO object at 0x7d129e825620>
Traceback (most recent call last):
  File "/usr/lib/python3.13/pathlib/_local.py", line 277, in _raw_path
    @property
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x7d129e7d18f0>
Traceback (most recent call last):
  File "/usr/lib/python3.13/pathlib/_local.py", line 277, in _raw_path
    @property
BufferError: Existing exports of data: object cannot be re-sized


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.2/30.2 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 4.8 MB/s eta 0:00:00
Environment successfully initialized!


In [19]:
# Modular Architectures (MLP + CNN) & Advanced PPO Agent

def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    nn.init.orthogonal_(layer.weight, std)
    nn.init.constant_(layer.bias, bias_const)
    return layer

class ActorNetwork(nn.Module):
    def __init__(self, n_actions, input_dims, is_continuous=False, is_visual=False):
        super(ActorNetwork, self).__init__()
        self.is_continuous = is_continuous
        self.is_visual = is_visual

        if self.is_visual:
            # Nature CNN Architecture for Visual Environments (e.g. Atari)
            self.network = nn.Sequential(
                layer_init(nn.Conv2d(input_dims[0], 32, 8, stride=4)),
                nn.ReLU(),
                layer_init(nn.Conv2d(32, 64, 4, stride=2)),
                nn.ReLU(),
                layer_init(nn.Conv2d(64, 64, 3, stride=1)),
                nn.ReLU(),
                nn.Flatten(),
                layer_init(nn.Linear(64 * 7 * 7, 512)),
                nn.ReLU()
            )
            fc_out_dim = 512
        else:
            # Standard MLP for Vector Environments (e.g. MuJoCo, LunarLander)
            self.network = nn.Sequential(
                layer_init(nn.Linear(*input_dims, 256)),
                nn.Tanh(),
                layer_init(nn.Linear(256, 256)),
                nn.Tanh()
            )
            fc_out_dim = 256

        if self.is_continuous:
            self.mu_head = layer_init(nn.Linear(fc_out_dim, n_actions), std=0.01)
            self.log_std_head = layer_init(nn.Linear(fc_out_dim, n_actions), std=0.01)
        else:
            self.action_head = layer_init(nn.Linear(fc_out_dim, n_actions), std=0.01)

    def forward(self, state):
        features = self.network(state)
        if self.is_continuous:
            mu = self.mu_head(features)
            log_std = T.clamp(self.log_std_head(features), min=-20, max=2)
            std = log_std.exp()
            return Normal(mu, std)
        else:
            logits = self.action_head(features)
            return Categorical(logits=logits)


class CriticNetwork(nn.Module):
    def __init__(self, input_dims, is_visual=False):
        super(CriticNetwork, self).__init__()
        self.is_visual = is_visual

        if self.is_visual:
            self.critic = nn.Sequential(
                layer_init(nn.Conv2d(input_dims[0], 32, 8, stride=4)),
                nn.ReLU(),
                layer_init(nn.Conv2d(32, 64, 4, stride=2)),
                nn.ReLU(),
                layer_init(nn.Conv2d(64, 64, 3, stride=1)),
                nn.ReLU(),
                nn.Flatten(),
                layer_init(nn.Linear(64 * 7 * 7, 512)),
                nn.ReLU(),
                layer_init(nn.Linear(512, 1), std=1.0)
            )
        else:
            self.critic = nn.Sequential(
                layer_init(nn.Linear(*input_dims, 256)),
                nn.Tanh(),
                layer_init(nn.Linear(256, 256)),
                nn.Tanh(),
                layer_init(nn.Linear(256, 1), std=1.0)
            )

    def forward(self, state):
        return self.critic(state)


class Agent:
    def __init__(
        self,
        n_actions,
        input_dims,
        gamma=0.99,
        alpha=3e-4,
        gae_lambda=0.95,
        policy_clip=0.2,
        batch_size=128,
        n_epochs=10,
        entropy_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        clip_vloss=True,
        is_continuous=False,
        is_visual=False,
        chkpt_dir='tmp/ppo'
    ):
        self.gamma = gamma
        self.alpha = alpha
        self.gae_lambda = gae_lambda
        self.policy_clip = policy_clip
        self.batch_size = batch_size
        self.n_epochs = n_epochs
        self.entropy_coef = entropy_coef
        self.vf_coef = vf_coef
        self.max_grad_norm = max_grad_norm
        self.clip_vloss = clip_vloss
        self.is_continuous = is_continuous
        self.is_visual = is_visual
        self.chkpt_dir = chkpt_dir

        os.makedirs(self.chkpt_dir, exist_ok=True)
        self.device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')

        self.actor = ActorNetwork(n_actions, input_dims, is_continuous, is_visual).to(self.device)
        self.critic = CriticNetwork(input_dims, is_visual).to(self.device)

        self.optimizer = optim.Adam(
            list(self.actor.parameters()) + list(self.critic.parameters()),
            lr=alpha,
            eps=1e-5
        )

    def choose_action(self, observation):
        state = T.tensor(observation, dtype=T.float32).to(self.device)
        with T.no_grad():
            dist = self.actor(state)
            value = self.critic(state)
            action = dist.sample()
            log_prob = dist.log_prob(action)
            if self.is_continuous:
                log_prob = log_prob.sum(axis=-1)
        return action.cpu().numpy(), log_prob.cpu().numpy(), value.squeeze(-1).cpu().numpy()

    def get_value(self, observation):
        state = T.tensor(observation, dtype=T.float32).to(self.device)
        with T.no_grad():
            value = self.critic(state).squeeze(-1)
        return value.cpu().numpy()

    def learn(self, memory_data, next_obs, writer=None, global_step=0):
        obs_b, actions_b, log_probs_b, rewards_b, terms_b, truncs_b, values_b = memory_data
        n_steps, n_envs = rewards_b.shape

        advantages = np.zeros_like(rewards_b, dtype=np.float32)
        next_value = self.get_value(next_obs)
        last_gae_lam = 0.0

        for t in reversed(range(n_steps)):
            if t == n_steps - 1:
                next_non_terminal = 1.0 - terms_b[t]
                next_values = next_value
            else:
                next_non_terminal = 1.0 - terms_b[t]
                next_values = values_b[t + 1]

            delta = rewards_b[t] + self.gamma * next_values * next_non_terminal - values_b[t]
            advantages[t] = last_gae_lam = delta + self.gamma * self.gae_lambda * next_non_terminal * last_gae_lam

        returns_b = advantages + values_b

        b_obs = T.tensor(obs_b.reshape((-1,) + obs_b.shape[2:]), dtype=T.float32).to(self.device)
        b_actions = T.tensor(actions_b.reshape((-1,) + actions_b.shape[2:])).to(self.device)
        b_log_probs = T.tensor(log_probs_b.reshape(-1), dtype=T.float32).to(self.device)
        b_advantages = T.tensor(advantages.reshape(-1), dtype=T.float32).to(self.device)
        b_returns = T.tensor(returns_b.reshape(-1), dtype=T.float32).to(self.device)
        b_values = T.tensor(values_b.reshape(-1), dtype=T.float32).to(self.device)

        batch_size = b_obs.shape[0]
        indices = np.arange(batch_size)

        a_losses, c_losses, e_losses, kl_divs = [], [], [], []

        for epoch in range(self.n_epochs):
            np.random.shuffle(indices)
            for start in range(0, batch_size, self.batch_size):
                end = start + self.batch_size
                mb_idx = indices[start:end]

                mb_obs = b_obs[mb_idx]
                mb_actions = b_actions[mb_idx]
                mb_old_log_probs = b_log_probs[mb_idx]
                mb_advantages = b_advantages[mb_idx]
                mb_returns = b_returns[mb_idx]
                mb_values = b_values[mb_idx]

                mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

                dist = self.actor(mb_obs)
                new_value = self.critic(mb_obs).squeeze(-1)

                new_log_probs = dist.log_prob(mb_actions)
                if self.is_continuous:
                    new_log_probs = new_log_probs.sum(axis=-1)

                log_ratio = new_log_probs - mb_old_log_probs
                ratio = log_ratio.exp()

                with T.no_grad():
                    approx_kl = ((ratio - 1) - log_ratio).mean().item()
                    kl_divs.append(approx_kl)

                pg_loss1 = -mb_advantages * ratio
                pg_loss2 = -mb_advantages * T.clamp(ratio, 1 - self.policy_clip, 1 + self.policy_clip)
                actor_loss = T.max(pg_loss1, pg_loss2).mean()

                if self.clip_vloss:
                    v_loss_unclipped = (new_value - mb_returns) ** 2
                    v_clipped = mb_values + T.clamp(new_value - mb_values, -self.policy_clip, self.policy_clip)
                    v_loss_clipped = (v_clipped - mb_returns) ** 2
                    critic_loss = 0.5 * T.max(v_loss_unclipped, v_loss_clipped).mean()
                else:
                    critic_loss = 0.5 * ((new_value - mb_returns) ** 2).mean()

                entropy_loss = dist.entropy().mean()
                total_loss = actor_loss + self.vf_coef * critic_loss - self.entropy_coef * entropy_loss

                self.optimizer.zero_grad()
                total_loss.backward()
                nn.utils.clip_grad_norm_(list(self.actor.parameters()) + list(self.critic.parameters()), self.max_grad_norm)
                self.optimizer.step()

                a_losses.append(actor_loss.item())
                c_losses.append(critic_loss.item())
                e_losses.append(entropy_loss.item())

        if writer is not None:
            writer.add_scalar('Loss/Actor', np.mean(a_losses), global_step)
            writer.add_scalar('Loss/Critic', np.mean(c_losses), global_step)
            writer.add_scalar('Diagnostics/KL_Divergence', np.mean(kl_divs), global_step)

In [20]:
# Environment Wrappers & PPO Runner Function

def make_env(env_id, is_visual=False):
    def thunk():
        if is_visual:
            env = gym.make(env_id, render_mode=None)
            env = gym.wrappers.AtariPreprocessing(env, screen_size=84, grayscale_obs=True, scale_obs=True)
            env = gym.wrappers.FrameStackObservation(env, stack_size=4)
        else:
            env = gym.make(env_id)
        env = gym.wrappers.RecordEpisodeStatistics(env)
        return env
    return thunk

def run_ppo_benchmark(env_id, total_timesteps=100000, n_envs=4, steps_per_env=128, is_visual=False):
    writer = SummaryWriter(f'runs/PPO_{env_id}_{int(time.time())}')
    envs = gym.vector.AsyncVectorEnv([make_env(env_id, is_visual) for _ in range(n_envs)])

    is_continuous = isinstance(envs.single_action_space, gym.spaces.Box)
    n_actions = envs.single_action_space.shape[0] if is_continuous else envs.single_action_space.n
    input_dims = envs.single_observation_space.shape

    agent = Agent(
        n_actions=n_actions,
        input_dims=input_dims,
        is_continuous=is_continuous,
        is_visual=is_visual,
        chkpt_dir=f'tmp/ppo_{env_id}'
    )

    n_steps_total = n_envs * steps_per_env
    num_updates = total_timesteps // n_steps_total

    obs_buf = np.zeros((steps_per_env, n_envs) + input_dims, dtype=np.float32)
    actions_buf = np.zeros((steps_per_env, n_envs) + envs.single_action_space.shape, dtype=np.float32)
    log_probs_buf = np.zeros((steps_per_env, n_envs), dtype=np.float32)
    rewards_buf = np.zeros((steps_per_env, n_envs), dtype=np.float32)
    terms_buf = np.zeros((steps_per_env, n_envs), dtype=np.bool_)
    truncs_buf = np.zeros((steps_per_env, n_envs), dtype=np.bool_)
    values_buf = np.zeros((steps_per_env, n_envs), dtype=np.float32)

    obs, _ = envs.reset()
    global_step = 0
    history_rewards = []
    history_steps = []

    print(f"\n---> Starting Custom PPO Training on [{env_id}] <---")

    for update in range(1, num_updates + 1):
        for step in range(steps_per_env):
            global_step += n_envs
            actions, log_probs, values = agent.choose_action(obs)
            exec_actions = actions if is_continuous else actions.squeeze()
            next_obs, rewards, dones, truncateds, infos = envs.step(exec_actions)

            obs_buf[step], actions_buf[step], log_probs_buf[step] = obs, actions, log_probs
            rewards_buf[step], terms_buf[step], truncs_buf[step], values_buf[step] = rewards, dones, truncateds, values

            obs = next_obs

            if "final_info" in infos:
                for info in infos["final_info"]:
                    if info and "episode" in info:
                        history_rewards.append(info["episode"]["r"][0])
                        history_steps.append(global_step)

        memory_data = (obs_buf, actions_buf, log_probs_buf, rewards_buf, terms_buf, truncs_buf, values_buf)
        agent.learn(memory_data, obs, writer, global_step)

        if update % max(1, (num_updates // 5)) == 0:
            avg_r = np.mean(history_rewards[-20:]) if history_rewards else 0.0
            print(f"Update {update:02d}/{num_updates} | Global Steps: {global_step} | Recent Avg Reward: {avg_r:.2f}")

    envs.close()
    writer.close()
    return history_steps, history_rewards

In [21]:
# Baseline Comparison (SAC / Stable-Baselines3) & Plotting
from stable_baselines3 import SAC, DQN
from stable_baselines3.common.callbacks import BaseCallback

class RewardTrackerCallback(BaseCallback):
    def __init__(self):
        super(RewardTrackerCallback, self).__init__()
        self.episode_rewards = []
        self.episode_steps = []

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.episode_rewards.append(info["episode"]["r"])
                self.episode_steps.append(self.num_timesteps)
        return True

# --- Select Benchmark Environment ---
# Options: 'HalfCheetah-v4' (MuJoCo Continuous), 'LunarLanderContinuous-v3', 'BreakoutNoFrameskip-v4'
TEST_ENV = 'HalfCheetah-v4'
IS_VISUAL = 'NoFrameskip' in TEST_ENV
TOTAL_TIMESTEPS = 100000

# 1. Train Custom PPO
ppo_steps, ppo_rewards = run_ppo_benchmark(TEST_ENV, total_timesteps=TOTAL_TIMESTEPS, is_visual=IS_VISUAL)

# 2. Train Off-Policy Baseline (SAC for Continuous Control)
print(f"\n---> Training Off-Policy Baseline (SAC) on [{TEST_ENV}] <---")
eval_env = gym.make(TEST_ENV)
eval_env = gym.wrappers.RecordEpisodeStatistics(eval_env)

sac_callback = RewardTrackerCallback()
baseline_model = SAC("MlpPolicy", eval_env, verbose=0)
baseline_model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=sac_callback)
eval_env.close()

# 3. Plotting & Benchmarking Comparison
plt.figure(figsize=(10, 6))

def smooth(y, box_pts=10):
    box = np.ones(box_pts)/box_pts
    return np.convolve(y, box, mode='same')

if len(ppo_rewards) > 0:
    plt.plot(ppo_steps, smooth(ppo_rewards), label='Custom PPO (On-Policy)', color='blue', linewidth=2)

if len(sac_callback.episode_rewards) > 0:
    plt.plot(sac_callback.episode_steps, smooth(sac_callback.episode_rewards), label='SAC Baseline (Off-Policy)', color='orange', linewidth=2)

plt.title(f'Performance Benchmark on {TEST_ENV}', fontsize=14)
plt.xlabel('Timesteps', fontsize=12)
plt.ylabel('Episode Reward', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=12)
plt.show()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/gymnasium/envs/registration.py:513: DeprecationWarning: WARN: The environment HalfCheetah-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/lib/python3.13/multiprocessing/popen_fork.py:73: DeprecationWarning: This process (pid=792) i


---> Starting Custom PPO Training on [HalfCheetah-v4] <---
Update 39/195 | Global Steps: 19968 | Recent Avg Reward: 0.00
Update 78/195 | Global Steps: 39936 | Recent Avg Reward: 0.00
Update 117/195 | Global Steps: 59904 | Recent Avg Reward: 0.00


KeyboardInterrupt: 